# Molecular Descriptors Calculator

Calculate RotB (Rotatable Bonds), tPSA (Topological Polar Surface Area), and LogP for a small set of compounds.

**Input format:** Dictionary with `ID: SMILES` pairs
**Output:** DataFrame with calculated descriptors

> **Note:** This notebook is standalone — it only imports RDKit directly, not the `py_utils` package.

In [1]:
# ═════════════════════════════════════════════════════════════════
# 1. DEFINE YOUR COMPOUNDS HERE (ID: SMILES)
# ═════════════════════════════════════════════════════════════════

compounds = {
    "Oxazolone":    "CC(OC/1=O)=NC1=C/C",
    "Amine":    "CN",
    "Carboxylic acid":    "CC(O)=O",
    "Aldehyde":   "CC([H])=O",
    "A20C25":   "O=C1NC(=O)N(Cc2ccccc2)C1=Cc1ccccc1",
    "A30C35":   "CCN(CC)C(=O)C1=NN(C)C(=O)N1Cc1ccc(O)cc1"
}

In [2]:
# ═════════════════════════════════════════════════════════════════
# 2. IMPORTS (RDKit only - no py_utils)
# ═════════════════════════════════════════════════════════════════

import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd

print(f"RDKit version: {rdkit.__version__}")
print(f"Compounds loaded: {len(compounds)}")

RDKit version: 2025.09.6
Compounds loaded: 6


In [3]:
# ═════════════════════════════════════════════════════════════════
# 3. CALCULATE DESCRIPTORS
# ═════════════════════════════════════════════════════════════════

def calculate_descriptors(smiles: str) -> dict:
    """
    Calculate RotB, tPSA, and LogP from a SMILES string.
    Returns dict with descriptors or None if SMILES is invalid.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Calculate descriptors
    RotB = Descriptors.NumRotatableBonds(mol)
    tPSA = Descriptors.TPSA(mol)
    LogP = Descriptors.MolLogP(mol)
    
    return {
        "RotB": RotB,
        "tPSA": tPSA,
        "LogP": LogP
    }


# Process all compounds
results = []

for comp_id, smiles in compounds.items():
    desc = calculate_descriptors(smiles)
    
    if desc is None:
        results.append({
            "ID": comp_id,
            "SMILES": smiles,
            "RotB": "INVALID",
            "tPSA": "INVALID",
            "LogP": "INVALID"
        })
    else:
        results.append({
            "ID": comp_id,
            "SMILES": smiles,
            "RotB": desc["RotB"],
            "tPSA": desc["tPSA"],
            "LogP": desc["LogP"]
        })

# Create DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*70)
print("MOLECULAR DESCRIPTORS CALCULATED")
print("="*70)
df_results


MOLECULAR DESCRIPTORS CALCULATED


,ID,SMILES,RotB,tPSA,LogP
0,Oxazolone,CC(OC/1=O)=NC1=C/C,0,38.66,0.8654
1,Amine,CN,0,26.02,-0.4251
2,Carboxylic acid,CC(O)=O,0,37.30,0.0909
3,Aldehyde,CC([H])=O,0,17.07,0.2052
4,A20C25,O=C1NC(=O)N(Cc2ccccc2)C1=Cc1ccccc1,3,49.41,2.7795
5,A30C35,CCN(CC)C(=O)C1=NN(C)C(=O)N1Cc1ccc(O)cc1,5,80.36,0.8177


In [4]:
# ═════════════════════════════════════════════════════════════════
# 4. SUMMARY STATISTICS
# ═════════════════════════════════════════════════════════════════

# Filter only valid results for statistics
df_valid = df_results[df_results["RotB"] != "INVALID"]

print("\n" + "="*70)
print("SUMMARY STATISTICS (only valid compounds)")
print("="*70)

summary = df_valid.describe()
print(summary[["RotB", "tPSA", "LogP"]])

print("\n" + "="*70)
print(f"Total compounds: {len(compounds)}")
print(f"Valid compounds: {len(df_valid)}")
print(f"Invalid compounds: {len(df_results) - len(df_valid)}")
print("="*70)


SUMMARY STATISTICS (only valid compounds)
           RotB       tPSA      LogP
count  6.000000   6.000000  6.000000
mean   1.333333  41.470000  0.722267
std    2.160247  22.067275  1.117518
min    0.000000  17.070000 -0.425100
25%    0.000000  28.840000  0.119475
50%    0.000000  37.980000  0.511450
75%    2.250000  46.722500  0.853475
max    5.000000  80.360000  2.779500

Total compounds: 6
Valid compounds: 6
Invalid compounds: 0
